In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
FS1 - TF-IDF Features Only
Trust-Based Explainable Fake News Detection System
=============================================================================

Feature Set : FS1
Engineering : TF-IDF Features Only
Meaning     : Uses word importance features only (unigrams, bigrams, trigrams)

This script uses ONLY TF-IDF vectorisation as the feature representation.
No linguistic or emotional/sentiment features are included.
This serves as the baseline textual-feature configuration.

Datasets
--------
Dataset 1:
    https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
Dataset 2:
    https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets

Software Requirements (Section 8.1)
-------------------------------------
    Python          >= 3.10
    scikit-learn    >= 1.2
    xgboost         >= 1.7
    pandas          >= 1.5
    numpy           >= 1.23
    nltk            >= 3.8
    shap            >= 0.41
    lime            >= 0.2
    matplotlib      >= 3.6
    seaborn         >= 0.12
=============================================================================
"""

# ---------------------------------------------------------------------------
# BLOCK 0 - PACKAGE INSTALLATION
# ---------------------------------------------------------------------------
import subprocess
import sys


def install_packages(packages):
    for package in packages:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", package]
        )
        print(f"Installed: {package}")


REQUIRED_PACKAGES = ["lime", "shap", "xgboost", "nltk"]
install_packages(REQUIRED_PACKAGES)


# ---------------------------------------------------------------------------
# BLOCK 1 - IMPORTS
# ---------------------------------------------------------------------------
import re
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import lime
import lime.lime_text

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from scipy.sparse import csr_matrix

try:
    from google.colab import drive
    COLAB_ENV = True
except ImportError:
    COLAB_ENV = False

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# BLOCK 2 - NLTK DOWNLOADS
# ---------------------------------------------------------------------------
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)


# ---------------------------------------------------------------------------
# BLOCK 3 - CONSTANTS AND CONFIGURATION
# ---------------------------------------------------------------------------
RANDOM_STATE     = 42
TEST_SIZE        = 0.20
CV_FOLDS         = 5
MAX_TFIDF_FEATURES = 3000  # Reduced so handcrafted features carry meaningful weight
NGRAM_RANGE      = (1, 3)
MIN_DF           = 2       # terms appearing in only one doc are usually noise
MAX_DF           = 0.95    # terms in almost every doc don't help distinguish classes
LABEL_FAKE       = 1
LABEL_REAL       = 0

FEATURE_SET_NAME = "FS1"
FEATURE_SET_DESC = "TF-IDF Features Only"

STOP_WORDS = set(stopwords.words("english"))


# ===========================================================================
# SECTION 1 - DATA LOADING AND PREPROCESSING
# ===========================================================================

def load_dataset(file_path, label):
    df = pd.read_csv(file_path)
    df["label"] = label
    df["title"] = df["title"].fillna("").astype(str)
    df["text"]  = df["text"].fillna("").astype(str)
    df = df[["title", "text", "label"]].drop_duplicates()
    logger.info("Loaded %d rows from '%s'.", len(df), file_path)
    return df


def combine_datasets(fake_path, real_path):
    df_fake = load_dataset(fake_path, label=LABEL_FAKE)
    df_real = load_dataset(real_path, label=LABEL_REAL)
    combined = (
        pd.concat([df_fake, df_real], ignore_index=True)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    logger.info(
        "Combined dataset: %d rows | Fake=%d | Real=%d",
        len(combined), combined["label"].sum(), (combined["label"] == 0).sum(),
    )
    return combined


def preprocess_text(raw_text):
    """
    Clean and normalise raw text with leakage-source removal.

    Extra steps vs baseline
    -----------------------
    Reuters dateline : Removes "CITY (Reuters) -" patterns present only in
                       real news. This is a near-perfect label signal that
                       causes TF-IDF alone to reach ~99% accuracy, making
                       FS1 through FS4 look identical. Removing it forces
                       the model to rely on actual content differences.
    Agency tags      : Strips (AP), (AFP), (BBC), (CNN) suffixes.
    """
    text = str(raw_text).lower()
    # --- leakage removal: Reuters dateline e.g. "washington (reuters) -" ---
    text = re.sub(r"\b\w[\w\s]*?\(reuters\)\s*-?\s*", " ", text)
    text = re.sub(r"\(ap\)|\(afp\)|\(bbc\)|\(cnn\)", " ", text)
    text = re.sub(r"-\s*reuters\b", " ", text)
    # --- standard cleaning ---
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    # 2-char cutoff removes leftover punctuation artifacts after regex stripping
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return " ".join(tokens)


# ===========================================================================
# SECTION 2 - FEATURE ENGINEERING  (FS1: TF-IDF Only)
# ===========================================================================

def compute_tfidf_features(train_corpus, test_corpus):
    """
    Fit TF-IDF on the training corpus and transform both sets.
    FS1 uses ONLY these features — no linguistic or emotional features added.
    """
    vectorizer = TfidfVectorizer(
        max_features=MAX_TFIDF_FEATURES,
        ngram_range=NGRAM_RANGE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        sublinear_tf=True,  # log(1+tf) instead of raw tf; stops very frequent terms from dominating
    )
    x_train_tfidf = vectorizer.fit_transform(train_corpus)
    x_test_tfidf  = vectorizer.transform(test_corpus)
    logger.info("[FS1] TF-IDF feature shape: %s", x_train_tfidf.shape)
    return x_train_tfidf, x_test_tfidf, vectorizer


# ===========================================================================
# SECTION 3 - SCORE DIAGNOSTICS
# ===========================================================================

def diagnose_scores(y_true, y_pred, y_prob, model_name):
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_true, y_prob)

    diagnostics = []

    if accuracy > 0.85 and f1 < 0.70:
        diagnostics.append(
            "WARNING: High accuracy but low F1 - possible class imbalance bias."
        )
    if precision > 0.85 and recall < 0.60:
        diagnostics.append(
            "WARNING: High precision with low recall - model is over-conservative."
        )
    if recall > 0.90 and precision < 0.60:
        diagnostics.append(
            "WARNING: High recall with low precision - too many false positives."
        )
    if roc_auc < 0.70:
        diagnostics.append(
            "WARNING: ROC-AUC below 0.70 - poor discriminatory ability."
        )
    if abs(accuracy - roc_auc) > 0.15:
        diagnostics.append(
            "INFO: Large gap between accuracy and ROC-AUC - consider threshold tuning."
        )
    if accuracy > 0.99 and f1 > 0.99:
        diagnostics.append(
            "CRITICAL: Near-perfect scores - check for data leakage."
        )

    for msg in diagnostics:
        logger.warning("[%s] %s", model_name, msg)

    return {
        "model":       model_name,
        "accuracy":    round(accuracy, 4),
        "precision":   round(precision, 4),
        "recall":      round(recall, 4),
        "f1_score":    round(f1, 4),
        "roc_auc":     round(roc_auc, 4),
        "diagnostics": diagnostics,
    }


# ===========================================================================
# SECTION 4 - MODEL DEFINITIONS
# ===========================================================================

def get_models():
    return [
        ("Naive Bayes",
         MultinomialNB(alpha=1.0)),          # Laplace smoothing; default works well for text
        ("Decision Tree",
         DecisionTreeClassifier(max_depth=20, min_samples_split=10,
                                random_state=RANDOM_STATE)),
        ("Logistic Regression",
         LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
                            random_state=RANDOM_STATE, n_jobs=-1)),
        ("K-Nearest Neighbour",
         KNeighborsClassifier(n_neighbors=3, metric="euclidean", n_jobs=-1)),
        ("Support Vector Machine",
         LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
        ("Random Forest",
         RandomForestClassifier(n_estimators=200, min_samples_split=5,
                                n_jobs=-1, random_state=RANDOM_STATE)),
        ("Gradient Boosting",
         GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                    max_depth=5, subsample=0.8,
                                    random_state=RANDOM_STATE)),
        ("XGBoost",
         XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                       subsample=0.8, colsample_bytree=0.8,
                       reg_alpha=0.1, reg_lambda=1.0,
                       eval_metric="logloss",
                       random_state=RANDOM_STATE, n_jobs=-1)),
    ]



def print_results_table(results_df, feature_set_name, feature_set_desc):
    """
    Print model results in a clean tabular format:
    Model | Accuracy | Precision | Recall | F1-score | ROC-AUC
    """
    col_order = ["model", "accuracy", "precision", "recall", "f1_score", "roc_auc"]
    display_headers = {
        "model":     "Model",
        "accuracy":  "Accuracy",
        "precision": "Precision",
        "recall":    "Recall",
        "f1_score":  "F1-score",
        "roc_auc":   "ROC-AUC",
    }
    df = results_df[col_order].copy()

    # Column widths
    col_widths = {
        "model":     26,
        "accuracy":  10,
        "precision": 11,
        "recall":    8,
        "f1_score":  10,
        "roc_auc":   9,
    }

    # Header
    header = (
        f"{'Model':<{col_widths['model']}}"
        f"{'Accuracy':>{col_widths['accuracy']}}"
        f"{'Precision':>{col_widths['precision']}}"
        f"{'Recall':>{col_widths['recall']}}"
        f"{'F1-score':>{col_widths['f1_score']}}"
        f"{'ROC-AUC':>{col_widths['roc_auc']}}"
    )
    sep = "-" * len(header)

    print(f"\n{'='*len(header)}")
    print(f"[{feature_set_name}] {feature_set_desc} - Model Performance Results")
    print(f"{'='*len(header)}")
    print(header)
    print(sep)

    for _, row in df.iterrows():
        print(
            f"{row['model']:<{col_widths['model']}}"
            f"{row['accuracy']:>{col_widths['accuracy']}.4f}"
            f"{row['precision']:>{col_widths['precision']}.4f}"
            f"{row['recall']:>{col_widths['recall']}.4f}"
            f"{row['f1_score']:>{col_widths['f1_score']}.4f}"
            f"{row['roc_auc']:>{col_widths['roc_auc']}.4f}"
        )
    print(sep)


# ===========================================================================
# SECTION 5 - TRAINING AND EVALUATION
# ===========================================================================

def train_and_evaluate(x_train, x_test, y_train, y_test, models):
    results = []

    for name, model in models:
        logger.info("[FS1] Training: %s", name)

        if name == "Naive Bayes":
            # MultinomialNB needs non-negative inputs; sublinear TF-IDF is always >=0
            # but clip anyway as a safeguard against any floating-point edge cases
            x_tr = x_train.copy()
            x_te = x_test.copy()
            x_tr.data = np.where(x_tr.data < 0, 0, x_tr.data)
            x_te.data = np.where(x_te.data < 0, 0, x_te.data)
        else:
            x_tr, x_te = x_train, x_test

        model.fit(x_tr, y_train)
        y_pred = model.predict(x_te)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(x_te)[:, 1]
        elif hasattr(model, "decision_function"):
            # LinearSVC has no predict_proba; min-max scale the raw margins to [0,1]
            # so we can still compute roc_auc_score
            scores = model.decision_function(x_te)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        else:
            y_prob = y_pred.astype(float)

        metrics = diagnose_scores(y_test, y_pred, y_prob, name)
        results.append(metrics)

    results_df = (
        pd.DataFrame(results)
        .drop(columns=["diagnostics"])
        .sort_values("f1_score", ascending=False)
        .reset_index(drop=True)
    )
    return results_df


# ===========================================================================
# SECTION 6 - HYPERPARAMETER TUNING
# ===========================================================================

def tune_best_model(x_train, y_train, best_model_name):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True,
                         random_state=RANDOM_STATE)

    param_grids = {
        "XGBoost": {
            "n_estimators":  [200, 300],
            "max_depth":     [4, 6],
            "learning_rate": [0.05, 0.1],
            "subsample":     [0.8],
        },
        "Random Forest": {
            "n_estimators":     [100, 200],
            "max_depth":        [None, 20],
            "min_samples_split":[2, 5],
        },
    }

    if best_model_name not in param_grids:
        logger.info("No grid for '%s'. Returning default.", best_model_name)
        _, model = next(m for m in get_models() if m[0] == best_model_name)
        model.fit(x_train, y_train)
        return model

    # run grid search on a 50% subsample to keep it tractable,
    # then refit the winning config on the full training set
    _, x_sub, _, y_sub = train_test_split(
        x_train, y_train, test_size=0.5,
        stratify=y_train, random_state=RANDOM_STATE,
    )
    _, base_model = next(m for m in get_models() if m[0] == best_model_name)
    grid = GridSearchCV(base_model, param_grids[best_model_name],
                        scoring="f1", cv=cv, n_jobs=-1, verbose=1)
    grid.fit(x_sub, y_sub)
    logger.info("Best params: %s | F1: %.4f",
                grid.best_params_, grid.best_score_)

    _, final_model = next(m for m in get_models() if m[0] == best_model_name)
    final_model.set_params(**grid.best_params_)
    final_model.fit(x_train, y_train)
    return final_model


# ===========================================================================
# SECTION 7 - EXPLAINABILITY
# ===========================================================================

def shap_global_explanation(model, x_train, feature_names, top_n=20):
    logger.info("[FS1] Computing SHAP global feature importance...")
    try:
        # TreeExplainer is the fast path for tree-based models (RF, XGBoost, GB)
        explainer  = shap.TreeExplainer(model)
        shap_vals  = explainer.shap_values(x_train)
        if isinstance(shap_vals, list):
            # older SHAP versions return [neg_class_shap, pos_class_shap]
            shap_vals = shap_vals[1]
    except Exception:
        # fall back to LinearExplainer for models like Logistic Regression
        background = shap.sample(x_train, 100)  # 100 samples balances speed vs accuracy
        explainer  = shap.LinearExplainer(model, background)
        shap_vals  = explainer.shap_values(x_train)

    mean_abs = np.abs(shap_vals).mean(axis=0)
    imp_df = (
        pd.DataFrame({"feature": feature_names, "importance": mean_abs})
        .sort_values("importance", ascending=False)
        .head(top_n)
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(data=imp_df, x="importance", y="feature",
                palette="viridis", ax=ax)
    ax.set_title(f"[FS1] SHAP Global Feature Importance - Top {top_n}")
    ax.set_xlabel("Mean |SHAP Value|")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.savefig("FS1_shap_global_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
    logger.info("Saved: FS1_shap_global_importance.png")


def lime_local_explanation(model, vectorizer, sample_text,
                           class_names=None):
    if class_names is None:
        class_names = ["Real News", "Fake News"]

    def predict_fn(texts):
        tfidf_part = vectorizer.transform(texts)
        return model.predict_proba(tfidf_part)

    explainer   = lime.lime_text.LimeTextExplainer(class_names=class_names)
    explanation = explainer.explain_instance(
        sample_text, predict_fn, num_features=15, num_samples=1000
    )
    explanation.show_in_notebook(text=True)
    explanation.save_to_file("FS1_lime_local_explanation.html")
    logger.info("Saved: FS1_lime_local_explanation.html")


def permutation_feature_importance(model, x_test, y_test,
                                   feature_names, top_n=20):
    logger.info("[FS1] Computing permutation feature importance...")
    baseline_acc = accuracy_score(y_test, model.predict(x_test))
    x_dense = x_test.toarray() if hasattr(x_test, "toarray") else x_test.copy()

    records = []
    # only permute the first top_n features — scanning all 3000 TF-IDF columns
    # would take too long and the most informative features tend to rank at the top
    for i, fname in enumerate(feature_names[:top_n]):
        x_perm = x_dense.copy()
        np.random.shuffle(x_perm[:, i])
        perm_acc = accuracy_score(y_test, model.predict(csr_matrix(x_perm)))
        records.append({"feature": fname,
                         "importance_drop": round(baseline_acc - perm_acc, 5)})

    return (
        pd.DataFrame(records)
        .sort_values("importance_drop", ascending=False)
        .reset_index(drop=True)
    )


# ===========================================================================
# SECTION 8 - STATISTICAL VALIDATION
# ===========================================================================

def statistical_validation(results_df, baseline_model="Logistic Regression"):
    rows = results_df.loc[
        results_df["model"] == baseline_model, "f1_score"
    ].values
    if len(rows) == 0:
        logger.warning("Baseline '%s' not found.", baseline_model)
        return results_df

    baseline_f1 = float(rows[0])
    records = []
    for _, row in results_df.iterrows():
        delta = round(row["f1_score"] - baseline_f1, 4)
        records.append({
            "model":                row["model"],
            "f1_score":             row["f1_score"],
            "vs_baseline_delta":    delta,
            "better_than_baseline": abs(delta) > 0.02 and delta > 0,  # 2-point gap as minimum meaningful difference
        })
    return pd.DataFrame(records)


# ===========================================================================
# SECTION 9 - VISUALISATION
# ===========================================================================

def plot_model_comparison(results_df):
    metrics = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
    melted  = results_df.melt(id_vars="model", value_vars=metrics,
                               var_name="Metric", value_name="Score")
    fig, ax = plt.subplots(figsize=(16, 7))
    sns.barplot(data=melted, x="model", y="Score",
                hue="Metric", palette="tab10", ax=ax)
    ax.set_title(f"[{FEATURE_SET_NAME}] Model Performance - {FEATURE_SET_DESC}")
    ax.set_xlabel("Model")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig("FS1_model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    logger.info("Saved: FS1_model_comparison.png")


def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"], ax=ax)
    ax.set_title(f"[FS1] Confusion Matrix - {model_name}")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    plt.tight_layout()
    safe = model_name.replace(" ", "_").lower()
    plt.savefig(f"FS1_confusion_matrix_{safe}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_roc_curves(models_dict, x_test, y_test):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey",
            label="Random Classifier (AUC=0.500)")

    for name, model in models_dict.items():
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(x_test)[:, 1]
        elif hasattr(model, "decision_function"):
            scores = model.decision_function(x_test)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        else:
            continue  # LinearSVC excluded from ROC — no probability output
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")

    ax.set_title("[FS1] ROC Curves - TF-IDF Only")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("FS1_roc_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    logger.info("Saved: FS1_roc_curves.png")


# ===========================================================================
# SECTION 10 - MAIN PIPELINE
# ===========================================================================

def run_pipeline(fake_csv_path, real_csv_path):
    """
    FS1 Pipeline - TF-IDF Features Only.

    Steps
    -----
    1.  Load and combine datasets.
    2.  Text preprocessing.
    3.  Stratified 80/20 train/test split.
    4.  TF-IDF feature extraction (ONLY feature set used).
    5.  Train and evaluate all 8 models.
    6.  Hyperparameter tuning of best model.
    7.  SHAP global explainability.
    8.  LIME local explainability.
    9.  Permutation feature importance.
    10. Statistical validation vs Logistic Regression baseline.
    11. Visualisation and CSV export.
    """
    logger.info("=== [FS1] STARTING PIPELINE: %s ===", FEATURE_SET_DESC)

    # Step 1 - Load
    logger.info("STEP 1: Loading datasets")
    df = combine_datasets(fake_csv_path, real_csv_path)
    # treat title + body as one document so the model sees the full article
    df["full_text"] = df["title"] + " " + df["text"]

    # Step 2 - Preprocess
    logger.info("STEP 2: Preprocessing text")
    df["clean_text"] = df["full_text"].apply(preprocess_text)

    # Step 3 - Split
    logger.info("STEP 3: Splitting data")
    # stratify keeps the class ratio the same in train and test
    train_clean, test_clean, train_raw, test_raw, y_train, y_test = (
        train_test_split(
            df["clean_text"], df["full_text"], df["label"],
            test_size=TEST_SIZE, stratify=df["label"],
            random_state=RANDOM_STATE,
        )
    )
    y_train, y_test = y_train.values, y_test.values
    logger.info("Train: %d | Test: %d", len(y_train), len(y_test))

    # Step 4 - TF-IDF ONLY
    logger.info("STEP 4: TF-IDF feature extraction (FS1 — no other features)")
    x_train_full, x_test_full, vectorizer = compute_tfidf_features(
        train_clean, test_clean
    )
    all_feature_names = vectorizer.get_feature_names_out().tolist()
    logger.info("[FS1] Total feature dimensions: %d", x_train_full.shape[1])

    # Step 5 - Train and evaluate
    logger.info("STEP 5: Training and evaluating models")
    models    = get_models()
    results_df = train_and_evaluate(
        x_train_full, x_test_full, y_train, y_test, models
    )
    print_results_table(results_df, FEATURE_SET_NAME, FEATURE_SET_DESC)

    # Step 6 - Hyperparameter tuning
    best_model_name = results_df.iloc[0]["model"]
    logger.info("STEP 6: Tuning best model: %s", best_model_name)
    tuned_model = tune_best_model(x_train_full, y_train, best_model_name)

    y_pred_tuned = tuned_model.predict(x_test_full)
    if hasattr(tuned_model, "predict_proba"):
        y_prob_tuned = tuned_model.predict_proba(x_test_full)[:, 1]
    else:
        scores = tuned_model.decision_function(x_test_full)
        y_prob_tuned = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    tuned_metrics = diagnose_scores(
        y_test, y_pred_tuned, y_prob_tuned,
        best_model_name + " (Tuned)"
    )
    print(f"\n[FS1] Tuned {best_model_name} Results:")
    for k, v in tuned_metrics.items():
        if k not in ("model", "diagnostics"):
            print(f"  {k}: {v}")
    if tuned_metrics["diagnostics"]:
        print("  Diagnostics:")
        for msg in tuned_metrics["diagnostics"]:
            print(f"    - {msg}")

    # Steps 7-9 - Explainability
    logger.info("STEPS 7-9: Explainability")
    if hasattr(tuned_model, "feature_importances_") or hasattr(tuned_model, "coef_"):
        shap_global_explanation(tuned_model, x_train_full,
                                all_feature_names, top_n=20)

    if hasattr(tuned_model, "predict_proba"):
        lime_local_explanation(tuned_model, vectorizer, test_raw.iloc[0])

    perm_imp = permutation_feature_importance(
        tuned_model, x_test_full, y_test, all_feature_names, top_n=20
    )
    print("\n[FS1] Top 10 Features by Permutation Importance:")
    print(perm_imp.head(10).to_string(index=False))

    # Step 10 - Statistical validation
    logger.info("STEP 10: Statistical validation")
    stats_df = statistical_validation(results_df)
    print("\n[FS1] Statistical Comparison vs Logistic Regression:")
    print(stats_df.to_string(index=False))

    # Step 11 - Visualisation and export
    logger.info("STEP 11: Visualisation and export")
    plot_model_comparison(results_df)
    plot_confusion_matrix(y_test, y_pred_tuned, best_model_name + " Tuned")

    trained_models = {}
    if hasattr(tuned_model, "predict_proba"):
        trained_models[best_model_name] = tuned_model
    for name, model in get_models():
        if name == best_model_name:
            continue
        if name == "Naive Bayes":
            x_tr = x_train_full.copy()
            x_tr.data = np.where(x_tr.data < 0, 0, x_tr.data)
            model.fit(x_tr, y_train)
            trained_models[name] = model
        elif name != "Support Vector Machine":
            model.fit(x_train_full, y_train)
            trained_models[name] = model

    plot_roc_curves(trained_models, x_test_full, y_test)

    results_df.to_csv("FS1_final_model_results.csv", index=False)
    logger.info("[FS1] Results saved to 'FS1_final_model_results.csv'.")
    logger.info("[FS1] Pipeline complete.")


# ===========================================================================
# ENTRY POINT
# ===========================================================================

if COLAB_ENV:
    drive.mount("/content/drive")
    FAKE_CSV = "/content/drive/MyDrive/Colab Notebooks/Dataset_2/dataset/Fake.csv"
    REAL_CSV = "/content/drive/MyDrive/Colab Notebooks/Dataset_2/dataset/True.csv"
# else:
#     FAKE_CSV = "data/Fake.csv"
#     REAL_CSV = "data/True.csv"

run_pipeline(fake_csv_path=FAKE_CSV, real_csv_path=REAL_CSV)
